In [1]:
from datasets import load_dataset, Dataset
from trl import SFTConfig, SFTTrainer
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainerCallback, DataCollatorWithPadding
import torch
from torch.utils.data import DataLoader
import random
from tqdm import tqdm

/workspace/llm-autoformalization/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
model_name = "AI-MO/Kimina-Autoformalizer-7B"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="cuda"
)

Loading weights: 100%|██████████| 339/339 [00:02<00:00, 143.24it/s]


In [3]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.padding_side = "left"

model.config.pad_token_id = tokenizer.pad_token_id
model.config.eos_token_id = tokenizer.eos_token_id
model.generation_config.pad_token_id = tokenizer.pad_token_id
model.generation_config.eos_token_id = tokenizer.eos_token_id

In [4]:
eval_dataset = load_dataset("PAug/ProofNetVerif")["test"]

In [5]:
PROMPT = """
Please autoformalize the following problem in Lean 4 with a header.
Use the following theorem names: my_favorite_theorem.\n\n
"""

In [6]:
def prepare_prompt(data: dict):
    header = data["lean4_src_header"].strip()
    problem = data["nl_statement"]

    messages = [
        {"role": "system", "content": "You are an expert in mathematics and Lean 4."},
        {"role": "user", "content": PROMPT + problem}
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    prefix = header + "\n" + "theorem my_favorite_theorem "
    prompt = text + "\n" + prefix

    return {"prompt": prompt}

In [7]:
eval_dataset = eval_dataset.map(prepare_prompt)

In [10]:
print(eval_dataset[1]["prompt"])

<|im_start|>system
You are an expert in mathematics and Lean 4.<|im_end|>
<|im_start|>user

Please autoformalize the following problem in Lean 4 with a header.
Use the following theorem names: my_favorite_theorem.


Prove that, for any integers $a, b, c$, there exists a positive integer $n$ such that $\sqrt{n^3+a n^2+b n+c}$ is not an integer.<|im_end|>
<|im_start|>assistant

import Mathlib

open scoped BigOperators
theorem my_favorite_theorem 


In [11]:
def tokenize_fn(example):
    return tokenizer(example["prompt"], truncation=True, max_length=512)

eval_dataset = eval_dataset.map(tokenize_fn)
eval_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask"]
)

In [12]:
collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    padding=True,
    return_tensors="pt",
)

loader = DataLoader(
    eval_dataset,
    batch_size=16,
    shuffle=False,
    collate_fn=collator,
)

In [40]:
all_answers = []

device = "cuda" if torch.cuda.is_available() else "cpu"
for batch in tqdm(loader):
    input_ids = batch["input_ids"].to(device)
    attention_mask = batch["attention_mask"].to(device)

    with torch.no_grad():
        outputs = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=256
        )

    generated_tokens = outputs[:, input_ids.shape[1]:].cpu()

    texts = tokenizer.batch_decode(
        generated_tokens,
        skip_special_tokens=True
    )

    all_answers.extend(texts)

100%|██████████| 91/91 [04:41<00:00,  3.10s/it]


In [45]:
final_answers = ["theorem my_favorite_theorem" + answer for answer in all_answers]

In [24]:
import pandas as pd

In [26]:
df = pd.read_csv("../benchmark_pred.csv")

In [36]:
print(df["lean4_formalization"].iloc[6])

theorem exercise_4_15 {f : ℝ → ℝ}
  (hf : Continuous f) (hof : IsOpenMap f) :
  Monotone f :=


In [46]:
df.shape

(1452, 10)

In [48]:
df["answers"] = final_answers

In [51]:
df.to_csv("../benchmark_pred.csv", index=False)

In [50]:
final_answers[0].removeprefix(df["lean4_src_header"].iloc[0])

'theorem my_favorite_theorem : Irreducible (X ^ 2 - C Zsqrtd : Polynomial (Zsqrtd)) :=\nsorry'

In [37]:
def prepare_formalization(
    formalization: str,
    header,
    add_sorry: bool,
    add_exact: bool
    ) -> str:
    formalization = formalization.removeprefix(header)
    formalization = formalization.rsplit(":=", 1)[0] + ":= by\n"

    if add_sorry:
        formalization += "sorry"
    elif add_exact:
        formalization += "exact?"
    
    return formalization

In [38]:
print(all_answers[100].rsplit(":=", 1)[0])

system
You are an expert in mathematics and Lean 4.
user

Please autoformalize the following problem in Lean 4 with a header.
Use the following theorem names: my_favorite_theorem.


Show that if $X$ is a countable product of spaces having countable dense subsets, then $X$ has a countable dense subset.
assistant

import Mathlib

open Filter Set TopologicalSpace
open scoped Topology
theorem my_favorite_theorem  {X : ℕ → Type*} [∀ i, TopologicalSpace (X i)]
  (h : ∀ i, ∃ (s : Set (X i)), Countable s ∧ Dense s) :
  ∃ (s : Set (Π i, X i)), Countable s ∧ Dense s 


In [39]:
print(all_answers[100])

system
You are an expert in mathematics and Lean 4.
user

Please autoformalize the following problem in Lean 4 with a header.
Use the following theorem names: my_favorite_theorem.


Show that if $X$ is a countable product of spaces having countable dense subsets, then $X$ has a countable dense subset.
assistant

import Mathlib

open Filter Set TopologicalSpace
open scoped Topology
theorem my_favorite_theorem  {X : ℕ → Type*} [∀ i, TopologicalSpace (X i)]
  (h : ∀ i, ∃ (s : Set (X i)), Countable s ∧ Dense s) :
  ∃ (s : Set (Π i, X i)), Countable s ∧ Dense s := by sorry


In [ ]:
prepare_formalization(
    
)